# Finding the Best Delivery Promise for Rosa's Pizza

In [59]:
!pip install -q git+https://github.com/zhouy185/rosa-starter.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [60]:
import numpy as np
from starter import ZONES, TIME_BLOCKS, COSTS, PROMISE, delivery_times

In [61]:
print("Zones:", ZONES)
print("Time blocks:", TIME_BLOCKS)
print("Costs:", COSTS)
print("Current promise:", PROMISE)

Zones: ['Central', 'North', 'Far West']
Time blocks: ['Lunch', 'Weekday eve', 'Fri/Sat eve', 'Other']
Costs: {'refund': 10.0, 'churn_orders': 1.8, 'margin': 9.0}
Current promise: 45


**Note**: A fixed seed of 1 is used for the simulation so that results remain consistent each time the sumulation is run.

In [81]:
times = delivery_times("Far West", "Fri/Sat eve", 45, seed=1)
print(len(times), "orders; average delivery time", round(times.mean(), 1),
"minutes")

210 orders; average delivery time 42.9 minutes


# Part 1: How bad is it, really?

### Part 1.a. Percentage of Late Orders

This function will be able to provide the percentage of orders that were delivered after the promised time of 45 minutes. It can provide a result given a specific zone and time-block or provide a result across all zones or time-blocks. The function requires 3 inputs: zone, time_block, promise.

In [92]:
def late_rate(zone, time_block, promise, seed=1):

    all_times = []

    if zone == "all":
        zones_to_use = ZONES
    else:
        zones_to_use = [zone]

    if time_block == "all":
        time_blocks_to_use = TIME_BLOCKS
    else:
        time_blocks_to_use = [time_block]

    for z in zones_to_use:
        for t in time_blocks_to_use:
            times = delivery_times(z, t, promise, seed=seed)
            all_times.extend(times)

    all_times = np.array(all_times)

    late_orders = np.sum(all_times > promise)

    total_orders = len(all_times)

    percentage_late = (late_orders / total_orders) * 100

    return percentage_late

In [96]:
# Case 1: Specific zone and specific time block
print("Case 1 - Far West, Fri/Sat eve:")
print(round(late_rate("Far West", "Fri/Sat eve", 45), 2), "%")


# Case 2: All zones and specific time block
print("Case 2 - All zones, Fri/Sat eve:")
print(round(late_rate("all", "Fri/Sat eve", 45), 2), "%")


# Case 3: Specific zone and all time blocks
print("Case 3 - Far West, All time blocks:")
print(round(late_rate("Far West", "all", 45), 2), "%")


# Case 4: All zones and all time blocks
print("Case 4 - All zones, All time blocks:")
print(round(late_rate("all", "all", 45), 2), "%")

Case 1 - Far West, Fri/Sat eve:
36.67 %
Case 2 - All zones, Fri/Sat eve:
20.91 %
Case 3 - Far West, All time blocks:
15.61 %
Case 4 - All zones, All time blocks:
8.29 %


Of the four Cases tested, case 1: Far West during Fri/Sat evening had the higest late rate of 36.67%. Case 4: All zones across all time-blocks had the lowest late rate of 8.29%. Due to the first three cases having a higher late rate than case 4; it signals that time-block Fri/Sat eve and zone Far West may be important to investigate further.

### Part 1.b. Ranking of Zones and Time-Blocks by Late Rate

In [99]:
late_rankings = []

for zone in ZONES:
    for time_block in TIME_BLOCKS:

        rate = late_rate(zone, time_block, 45, seed=1)

        late_rankings.append((zone, time_block, rate))

late_rankings.sort(key=lambda x: x[2], reverse=True)

for zone, time_block, rate in late_rankings:
    print(zone, "|", time_block, "|", round(rate, 2), "%")

Far West | Fri/Sat eve | 36.67 %
North | Fri/Sat eve | 16.67 %
Far West | Weekday eve | 11.18 %
Central | Fri/Sat eve | 6.84 %
North | Weekday eve | 6.0 %
Far West | Other | 4.69 %
Central | Weekday eve | 4.24 %
North | Lunch | 3.68 %
Central | Lunch | 3.46 %
Central | Other | 2.63 %
North | Other | 2.5 %
Far West | Lunch | 2.5 %


This ranking shows us that the highest rate of late deliveries is in the Far West zone during Fri/Sat evening at 36.67 %. That is substantially higher than all other zone and time-block combinations, the next ranking is North during Fri/Sat evening at 16.67%. The ranking also shows that within each zone the highest late arrival rate is during Fri/Sat evening. This suggests that the Far West zone and Fri/Sat evening time-block experience the highest rate of late deliveries.

### Part 1.c. Average Delivery Times

This function will calculate the average delivery time given a specific zone and time-block or provide a result across all zones or time-blocks.

In [101]:
def average_delivery_time(zone, time_block, promise, seed=1):

    all_times = []

    if zone == "all":
        zones_to_use = ZONES
    else:
        zones_to_use = [zone]

    if time_block == "all":
        time_blocks_to_use = TIME_BLOCKS
    else:
        time_blocks_to_use = [time_block]

    for z in zones_to_use:
        for t in time_blocks_to_use:
            times = delivery_times(z, t, promise, seed=seed)
            all_times.extend(times)

    all_times = np.array(all_times)

    average_time = np.mean(all_times)

    return average_time

In [103]:
# Case 1: Specific zone and specific time block
print("Case 1 - Far West, Fri/Sat eve:",
      round(average_delivery_time("Far West", "Fri/Sat eve", 45), 2),
      "minutes")

# Case 2: All zones and specific time block
print("Case 2 - All zones, Fri/Sat eve:",
      round(average_delivery_time("all", "Fri/Sat eve", 45), 2),
      "minutes")

# Case 3: Specific zone and all time blocks
print("Case 3 - Far West, All time blocks:",
      round(average_delivery_time("Far West", "all", 45), 2),
      "minutes")

# Case 4: All zones and all time blocks
print("Case 4 - All zones, All time blocks:",
      round(average_delivery_time("all", "all", 45), 2),
      "minutes")

Case 1 - Far West, Fri/Sat eve: 42.92 minutes
Case 2 - All zones, Fri/Sat eve: 37.51 minutes
Case 3 - Far West, All time blocks: 35.27 minutes
Case 4 - All zones, All time blocks: 31.19 minutes


Of the four Cases tested, case 1: Far West during Fri/Sat evening had the higest average delivery time of 42.92 minutes. Case 4: All zones across all time-blocks had the lowest average delivery time of 31.19 minutes. When comparing Fri/Sat evening across all zones and Far West across all time-blocks, Fri/Sat evening across all zones has a higher average delivery time than Far West across all time-blocks.

### Part 1.d. Ranking of Zones and Time-Blocks by Average Delivery Time

In [105]:
average_rankings = []

for zone in ZONES:
    for time_block in TIME_BLOCKS:

        avg_time = average_delivery_time(zone, time_block, 45, seed=1)

        average_rankings.append((zone, time_block, avg_time))

average_rankings.sort(key=lambda x: x[2], reverse=True)

for zone, time_block, avg_time in average_rankings:
    print(zone, "|", time_block, "|", round(avg_time, 2), "minutes")

Far West | Fri/Sat eve | 42.92 minutes
North | Fri/Sat eve | 36.77 minutes
Far West | Weekday eve | 35.04 minutes
Central | Fri/Sat eve | 32.13 minutes
North | Weekday eve | 31.38 minutes
Far West | Lunch | 30.58 minutes
Far West | Other | 30.04 minutes
Central | Weekday eve | 29.03 minutes
North | Lunch | 28.67 minutes
North | Other | 27.66 minutes
Central | Lunch | 26.8 minutes
Central | Other | 26.04 minutes


This ranking shows us that the Far West zone during Fri/Sat evening has the highest average delivery time at 42.92 minutes. Far West also shows at the top ranking across other time-blocks. Similary, Fri/Sat evening shows near the top of the ranking across other zones. The central zone seems to have the lowest average delivery times across most zones.

### Part 1.e. Which Measure is More Relevant?

The rates of late deliveries is more relevant to Rosa's decision because it shows how often orders are no able to meet the 45 minute delivery guarantee across all zones and time-blocks. Average delivery time does not show the number of orders that are delivered late. Another trade-off of average delivery times is that faster delivery times can offset longer longer delivery times creating an averages that are still under the promised time of 45 minutes. For example the average delivery time in the Far West during Fri/Sat evening is just under the promised time, however the late rate function tells us that almost half the orders are late in that zone and time-block. The late rate function is more informative in creating a clearer picture for how often the delivery time goes over the promised time in each zone and time-block and is therefore more useful for making decisions.

## Part 2: Exploring the Best Promise

### Part 2.a. Cost Per Late Order

This function will calculate the total cost per late order. This function takes into account the direct refund cost and indirect cost from customer churn. Churn is calculated by multiplying the estimate of future orders lost by the profit margin per oder.

In [69]:
print(COSTS)

{'refund': 10.0, 'churn_orders': 1.8, 'margin': 9.0}


In [70]:
def cost_per_late_order(costs):
    refund = costs["refund"]
    churn_cost = costs["churn_orders"] * costs["margin"]

    total_cost = refund + churn_cost

    return total_cost

In [71]:
print("Cost per late order: $", round(cost_per_late_order(COSTS), 2))

Cost per late order: $ 26.2


Each late order costs Rosa an estimated $26.20 in future profit.

### Part 2.b. Delivery Promises

This funtion will determine what delivery promise time will result in the most profitable outcome. The four inputs of the function are zone, time_block, promises, and costs.

In [72]:
promises = list(range(20, 71, 5))
print(promises)

[20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70]


The list of promise times is chosen based on the low end of the average delivery times in all zones across all time-blocks and then goes up by five minute increments. Multiple promise times higher than the current promise time are being tested as almost half the deliveries in the Far West zone during Fri/Sat evening are late. This will allow us to see if extended the promise time to something more achievable could result in higher net profit.

In [106]:
def find_best_promise(zone, time_block, promises, costs, seed=1):

    late_cost = cost_per_late_order(costs)

    best_promise_time = None
    best_net_profit = -np.inf

    results = []

    for promise in promises:

        times = delivery_times(zone, time_block, promise, seed=seed)

        number_of_orders = len(times)

        late_orders = np.sum(times > promise)

        profit_collected = number_of_orders * costs["margin"]

        total_late_cost = late_orders * late_cost

        net_profit = profit_collected - total_late_cost

        results.append(
            (promise, number_of_orders, late_orders, net_profit)
        )

        if net_profit > best_net_profit:
            best_net_profit = net_profit
            best_promise_time = promise

    return best_promise_time, best_net_profit, results

In [108]:
best_time, best_profit, results = find_best_promise(
    "Far West",
    "Fri/Sat eve",
    promises,
    COSTS,
    seed=1
)

print("Recommended promise:", best_time, "minutes")
print("Net profit: $", round(best_profit, 2))

Recommended promise: 55 minutes
Net profit: $ 1212.4


In [109]:
for promise, orders, late_orders, net_profit in results:
    print(
        promise, "minutes |",
        orders, "orders |",
        late_orders, "late orders |",
        "$", round(net_profit, 2), "net profit"
    )

20 minutes | 241 orders | 239 late orders | $ -4092.8 net profit
25 minutes | 239 orders | 233 late orders | $ -3953.6 net profit
30 minutes | 237 orders | 218 late orders | $ -3578.6 net profit
35 minutes | 232 orders | 179 late orders | $ -2601.8 net profit
40 minutes | 224 orders | 133 late orders | $ -1468.6 net profit
45 minutes | 210 orders | 77 late orders | $ -127.4 net profit
50 minutes | 188 orders | 33 late orders | $ 827.4 net profit
55 minutes | 158 orders | 8 late orders | $ 1212.4 net profit
60 minutes | 121 orders | 2 late orders | $ 1036.6 net profit
65 minutes | 84 orders | 0 late orders | $ 756.0 net profit
70 minutes | 54 orders | 0 late orders | $ 486.0 net profit


In [111]:
best_time, best_profit, results = find_best_promise(
    "North",
    "Fri/Sat eve",
    promises,
    COSTS,
    seed=1
)

print("Recommended promise:", best_time, "minutes")
print("Net profit: $", round(best_profit, 2))

Recommended promise: 50 minutes
Net profit: $ 996.4


In [113]:
for promise, orders, late_orders, net_profit in results:
    print(
        promise, "minutes |",
        orders, "orders |",
        late_orders, "late orders |",
        "$", round(net_profit, 2), "net profit"
    )

20 minutes | 172 orders | 169 late orders | $ -2879.8 net profit
25 minutes | 171 orders | 160 late orders | $ -2653.0 net profit
30 minutes | 169 orders | 133 late orders | $ -1963.6 net profit
35 minutes | 166 orders | 101 late orders | $ -1152.2 net profit
40 minutes | 160 orders | 59 late orders | $ -105.8 net profit
45 minutes | 150 orders | 25 late orders | $ 695.0 net profit
50 minutes | 134 orders | 8 late orders | $ 996.4 net profit
55 minutes | 113 orders | 3 late orders | $ 938.4 net profit
60 minutes | 87 orders | 0 late orders | $ 783.0 net profit
65 minutes | 60 orders | 0 late orders | $ 540.0 net profit
70 minutes | 39 orders | 0 late orders | $ 351.0 net profit


The first test of the function for the Far West during Fri/Sat evening shows that net profit increases as the promised delivery time is extended past the current promise time of 45 minutes till 55 minutes, where net profit is $1212.40. After 55 minutes net profit declines. This is 10 minutes longer than Rosa's current promise time of 45 minutes, suggesting that extending the promised time could improve net profit by reducing the costs from late orders.

The second test for North zone during Fri/Sat evening produces a similar result. Net profit increased till the 50 minute mark, where net profit is $996.40, then declines past that promise time.

